# Benchmarking LLMs with MMLU

Generic metrics like accuracy or F1 tell us *how* well a model classifies, but they don't tell us *what* a language model actually knows.  For that we use **benchmarks**: curated collections of tasks designed to probe specific capabilities.

Two widely cited benchmarks are:

* **GLUE** (General Language Understanding Evaluation): a suite of natural language understanding tasks — sentiment classification, paraphrase detection, textual entailment, and so on.  Each subtask uses its own metric (accuracy, F1, correlation), and the overall GLUE score is the average across subtasks.
* **MMLU** (Massive Multitask Language Understanding): 57 multiple-choice exam-style subject subsets, from elementary mathematics to professional law.  The metric is simply accuracy, averaged over the subsets.

Because MMLU is just multiple-choice question answering scored with accuracy, the mechanics are very transparent.  In this notebook we'll work through the whole MMLU evaluation pipeline by hand:

* load a small slice of the dataset,
* format the questions into prompts the model can answer,
* run a model and extract its A/B/C/D answer,
* compute the per-subset accuracy,
* average across subsets to produce a final score.

## The MMLU dataset

The MMLU dataset is hosted on the Hugging Face Hub under `cais/mmlu` and can be loaded with the `datasets` library.  If you don't have it installed yet:

```bash
pip install datasets
```

In [ ]:
from datasets import load_dataset

MMLU is split into 57 named subject subsets (e.g. `abstract_algebra`, `anatomy`, `high_school_physics`, ...).  Each subset has its own `test`, `validation`, and `dev` splits.

To keep things fast while we're exploring, we'll grab just the first few questions of the `test` split for one subject:

In [ ]:
subset_name = "abstract_algebra"
num_questions = 5
ds = load_dataset("cais/mmlu", subset_name, split=f"test[:{num_questions}]")

Let's see what fields each row has:

In [ ]:
ds.features

The relevant fields are:

* `question`: the question text (a string)
* `choices`: a list of four answer options
* `answer`: an integer 0, 1, 2, or 3 indicating the correct choice

Let's look at the first row:

In [ ]:
ds[0]

And just to see each field in isolation:

In [ ]:
ds[0]['question']

In [ ]:
ds[0]['choices']

In [ ]:
ds[0]['answer']

## Building a prompt

To ask an LLM a multiple-choice question, we need to format the question and its choices into a prompt and tell the model how to answer.  The standard MMLU convention is to label the four choices `A`, `B`, `C`, `D` (matching indices 0, 1, 2, 3) and ask the model to respond with a single letter.

In [ ]:
OPTION_LETTERS = ["A", "B", "C", "D"]

A helper function that turns a question and its four choices into a complete prompt:

In [ ]:
def build_prompt(question, choices):
    options_text = "\n".join(
        f"{letter}. {choice}"
        for letter, choice in zip(OPTION_LETTERS, choices)
    )
    return f"""You are an expert exam-taker.

Question:
{question}

Options:
{options_text}

Answer with just the letter: A, B, C, or D.
"""

Let's see what a prompt looks like for one of the questions:

In [ ]:
print(build_prompt(ds[0]['question'], ds[0]['choices']))

## A random-guessing baseline

Before plugging in a real LLM, it's useful to know what *random* performance looks like.  With four choices, a model that just guesses uniformly should get about 25% accuracy in the long run.  Anything well above 25% suggests the model actually knows something.

In [ ]:
import random

In [ ]:
def random_guess(prompt):
    """Stand-in for a real LLM: just picks A/B/C/D at random."""
    return random.choice(OPTION_LETTERS)

In [ ]:
random_guess(ds[0]['question'])

Now the scoring loop.  For each row we:

1. build the prompt,
2. ask the (placeholder) model for an answer,
3. scan the model's output for the first valid letter,
4. compare that letter (mapped back to an index 0–3) against the gold answer,
5. tally correct answers and divide by the total.

In [ ]:
correct = 0
total = len(ds)

for row in ds:
    question = row["question"]
    choices = row["choices"]
    gold_idx = row["answer"]

    prompt = build_prompt(question, choices)
    output = random_guess(prompt).strip().upper()

    # Grab the first valid letter in the output
    pred_letter = None
    for ch in output:
        if ch in OPTION_LETTERS:
            pred_letter = ch
            break

    if pred_letter is None:
        # Model didn't give a valid letter; count as wrong
        continue
    
    pred_idx = OPTION_LETTERS.index(pred_letter)

    if pred_idx == gold_idx:
        correct += 1

accuracy = correct / total if total > 0 else 0.0
print(f"{subset_name}: {correct}/{total} correct  ->  accuracy = {accuracy:.3f}")

With only 5 questions this number is noisy — try re-running it a few times and you'll see it jump around.  With more questions it will converge toward 0.25, which is the random baseline.

## Plugging in a real LLM

Now we replace `random_guess` with an actual language model.  Here we use `llama-cpp-python` to run a small quantized Llama 3.1 model locally, but the same scoring code works with any function that takes a prompt string and returns a text response (an OpenAI API call, a Hugging Face `pipeline`, etc.).

In [ ]:
from llama_cpp import Llama

import logging
logging.getLogger("llama_cpp").setLevel(logging.ERROR)

In [ ]:
# These are shell commands to download GGUF files 
# Uncomment as needed
#
# !hf download bartowski/Meta-Llama-3.1-8B-Instruct-GGUF --include "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf" --local-dir ./
# !hf download bartowski/Llama-3.2-1B-Instruct-GGUF --include "Llama-3.2-1B-Instruct-Q4_K_M.gguf" --local-dir ./

Load the model.  `n_ctx` is the context window in tokens and `n_threads` should be set to roughly the number of physical CPU cores.

In [ ]:
llm = Llama(
    model_path="Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    # model_path="Llama-3.2-1B-Instruct-Q4_K_M.gguf",
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

A quick sanity check — pass one of our prompts directly to the model.  We use a low `temperature` so the answers are nearly deterministic, and `max_tokens=3` because we only need a letter (a couple of tokens of slack in case the model adds whitespace).

In [ ]:
llm(
    build_prompt(ds[0]['question'], ds[0]['choices']),
    temperature=0.1,
    max_tokens=3,
)

The return value is a dict; the generated text is at `response['choices'][0]['text']`.  Let's wrap that in a function with the same signature as our random baseline so we can drop it into the same scoring loop:

In [ ]:
def real_llm(prompt):
    response = llm(prompt, temperature=0.1, max_tokens=3)
    return response['choices'][0]['text']

In [ ]:
real_llm(build_prompt(ds[0]['question'], ds[0]['choices']))

Now we run the same scoring loop as before, but call `real_llm` instead of `random_guess`:

In [ ]:
correct = 0
total = len(ds)

for row in ds:
    question = row["question"]
    choices = row["choices"]
    gold_idx = row["answer"]

    prompt = build_prompt(question, choices)
    output = real_llm(prompt).strip().upper()

    pred_letter = None
    for ch in output:
        if ch in OPTION_LETTERS:
            pred_letter = ch
            break

    if pred_letter is None:
        continue

    pred_idx = OPTION_LETTERS.index(pred_letter)

    if pred_idx == gold_idx:
        correct += 1

accuracy = correct / total if total > 0 else 0.0
print(f"{subset_name}: {correct}/{total} correct  ->  accuracy = {accuracy:.3f}")

## Chat-style prompting

Instruction-tuned models like Llama 3.1 Instruct are trained on chat-formatted inputs (with a system message and one or more user/assistant turns).  Treating the prompt as a raw completion (as we just did) usually works, but using the chat format tends to give cleaner, better-behaved answers.

`llama-cpp-python` exposes this through `create_chat_completion`:

In [ ]:
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Use one letter answers."},
        {"role": "user",   "content": build_prompt(ds[0]['question'], ds[0]['choices'])},
    ],
    temperature=0.1,
)
response

The generated text now lives at `response['choices'][0]['message']['content']` (note `message` instead of `text`):

In [ ]:
response['choices'][0]['message']['content']

In [ ]:
def real_llm_oneletter(prompt):
    response = llm.create_chat_completion(
                    messages=[
                        {"role": "system", "content": "Use one letter answers."},
                        {"role": "user",   "content": prompt},
                    ],
                temperature=0.1,
                )
    return response['choices'][0]['message']['content']

In [ ]:
real_llm_oneletter(build_prompt(ds[0]['question'], ds[0]['choices']))

## Bundling into a reusable function

A real MMLU score is the average accuracy across all 57 subject subsets.  We don't need to do all 57 here, but the structure is exactly the same as what we've built: load a subset, loop over rows, score, average.

Here's the whole per-subset pipeline wrapped into one function:

In [ ]:
def eval_mmlu_subset(subset_name="abstract_algebra", num_questions=20):
    """Evaluate the model on one MMLU subject subset and return accuracy."""
    ds = load_dataset("cais/mmlu", subset_name, split=f"test[:{num_questions}]")

    correct = 0
    total = len(ds)

    for row in ds:
        question = row["question"]
        choices = row["choices"]
        gold_idx = row["answer"]

        prompt = build_prompt(question, choices)
        # output = random_guess(prompt).strip().upper()   # our variation of model
        # output = real_llm(prompt).strip().upper()   
        output = real_llm_oneletter(prompt).strip().upper()   

        pred_letter = None
        for ch in output:
            if ch in OPTION_LETTERS:
                pred_letter = ch
                break

        if pred_letter is None:
            continue

        pred_idx = OPTION_LETTERS.index(pred_letter)

        if pred_idx == gold_idx:
            correct += 1

    accuracy = correct / total if total > 0 else 0.0
    print(f"{subset_name}: {correct}/{total} correct  ->  accuracy = {accuracy:.3f}")
    return accuracy

Run it across a couple of subsets and average to get an MMLU-style score:

In [ ]:
subsets = ["abstract_algebra", "anatomy"]
scores = [eval_mmlu_subset(s, num_questions=20) for s in subsets]

mmlu_score = sum(scores) / len(scores)
print(f"\nMMLU score over {len(subsets)} subsets: {mmlu_score:.3f}")

## And... with one of our NRP AI models too for fun

In [ ]:
from openai import OpenAI

# This assumes that you have a "keys.py" file in this directory
# with NRP_TOK assigned the value of your NRP API token (required)
# and NRP_CACHE_SALT assigned your cache_salt value (optional)
import keys
NRP_TOK = keys.NRP_TOK
NRP_CACHE_SALT = keys.NRP_CACHE_SALT

llm_client = OpenAI(api_key = NRP_TOK,
                    base_url = "https://ellm.nrp-nautilus.io/v1")
nrp_chat_model = 'gpt-oss'

In [ ]:
def real_llm_api(prompt):
    response = llm_client.chat.completions.create(
                    model=nrp_chat_model,
                    messages=[
                        {"role": "system", "content": "Use one letter answers."},
                        {"role": "user",   "content": prompt},
                    ],
                    temperature=0.1,
                    extra_body={"cache_salt": NRP_CACHE_SALT}
                )
    return response.choices[0].message.content

In [ ]:
real_llm_api(build_prompt(ds[0]['question'], ds[0]['choices']))

In [ ]:
def eval_mmlu_subset(subset_name="abstract_algebra", num_questions=20):
    """Evaluate the model on one MMLU subject subset and return accuracy."""
    ds = load_dataset("cais/mmlu", subset_name, split=f"test[:{num_questions}]")

    correct = 0
    total = len(ds)

    for row in ds:
        question = row["question"]
        choices = row["choices"]
        gold_idx = row["answer"]

        prompt = build_prompt(question, choices)
        # output = random_guess(prompt).strip().upper()   # our variation of model
        # output = real_llm(prompt).strip().upper()   
        # output = real_llm_oneletter(prompt).strip().upper()   
        output = real_llm_api(prompt).strip().upper()   

        pred_letter = None
        for ch in output:
            if ch in OPTION_LETTERS:
                pred_letter = ch
                break

        if pred_letter is None:
            continue

        pred_idx = OPTION_LETTERS.index(pred_letter)

        if pred_idx == gold_idx:
            correct += 1

    accuracy = correct / total if total > 0 else 0.0
    print(f"{subset_name}: {correct}/{total} correct  ->  accuracy = {accuracy:.3f}")
    return accuracy

Run it across a couple of subsets and average to get an MMLU-style score:

In [ ]:
subsets = ["abstract_algebra", "anatomy"]
scores = [eval_mmlu_subset(s, num_questions=20) for s in subsets]

mmlu_score = sum(scores) / len(scores)
print(f"\nMMLU score over {len(subsets)} subsets: {mmlu_score:.3f}")

For a **full** MMLU score you would:

* loop over all 57 subject subsets instead of just two,
* use the full `test` split rather than the first 20 questions,
* often use **few-shot** prompts (a handful of solved example questions prepended to the prompt) instead of pure zero-shot.

…but the structure (load → prompt → choose A/B/C/D → compute accuracy → average) stays exactly the same.